# 3 Ways to Add Tools to a LangChain Agent
1. **Function as a Tool** - Plain Python function
2. **Inbuilt/Community Tool** - Pre-built tools from LangChain community
3. **Agent as a Tool** - Another agent wrapped as a tool

In [ ]:
!pip install langchain langchain-openai langchain-community wikipedia

In [ ]:
from google.colab import userdata
import os
os.environ["AZURE_OPENAI_API_KEY"] = userdata.get('AZURE_OPENAI_API_KEY')
os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get('AZURE_OPENAI_ENDPOINT')
os.environ["OPENAI_API_VERSION"] = "2025-03-01-preview"

In [ ]:
from langchain_openai import AzureChatOpenAI

model = AzureChatOpenAI(
    model="gpt-4.1-mini",
    azure_deployment="gpt-4.1-mini"
)

## 1. Function as a Tool
A plain Python function with type hints and docstring is automatically converted to a tool.

In [ ]:
from langchain.agents import create_agent

def multiply(a: int, b: int) -> int:
    """Multiply two numbers and return the result."""
    return a * b

agent1 = create_agent(
    model,
    tools=[multiply],
    system_prompt="You are a calculator assistant. Use the multiply tool when asked to multiply numbers."
)

response = agent1.invoke({"messages": [{"role": "user", "content": "What is 7 times 8?"}]})
print(response['messages'][-1].content)

## 2. Inbuilt/Community Tool
LangChain provides pre-built tools like Wikipedia, DuckDuckGo, etc.

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500))

agent2 = create_agent(
    model,
    tools=[wiki_tool],
    system_prompt="You are a research assistant. Use the wikipedia tool to answer questions."
)

response = agent2.invoke({"messages": [{"role": "user", "content": "Tell me about the Eiffel Tower"}]})
print(response['messages'][-1].content)

## 3. Agent as a Tool
Wrap another agent inside a tool function so the main agent can delegate tasks to it.

In [ ]:
from langchain.tools import tool

# Create a sub-agent that handles math
math_agent = create_agent(
    model,
    tools=[multiply],
    system_prompt="You are a math expert. Solve the math problem using available tools and return ONLY the final answer."
)

# Wrap the sub-agent as a tool
@tool
def ask_math_agent(question: str) -> str:
    """Delegate math questions to a specialized math agent."""
    response = math_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return response["messages"][-1].content

# Main agent uses the sub-agent as a tool alongside wiki
agent3 = create_agent(
    model,
    tools=[wiki_tool, ask_math_agent],
    system_prompt="You are a general assistant. Use wikipedia for knowledge questions and ask_math_agent for calculations."
)

response = agent3.invoke({"messages": [{"role": "user", "content": "What is 15 times 23? Also tell me who built the Eiffel Tower."}]})
print(response['messages'][-1].content)